# 02 - 9-Level Strategy Chain in Detail

> **When to use**: When you need to customize the data type per column, or are unsatisfied with auto-inferred results.
>
> **Core concept**: sqlseed's `ColumnMapper` auto-matches column names → generators by 9-level priority. Understanding this chain lets you precisely control each column's data.

## Applicable Scenarios

- Column names are non-standard (e.g., `user_name` instead of `name`), need manual generator assignment
- Need to limit data range (e.g., `age` between 18-65)
- Need to generate specific pattern data (e.g., order number `ORD-\d{6}`)
- Want to understand sqlseed's auto-inference logic

## What You Will Learn

- Complete priority of the 9-level strategy chain
- Matching logic and trigger conditions for each level
- 75 exact match rules + 29 regex patterns
- How to override auto-inference with `columns={}`

See architecture.zh-CN.md §3

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| **→ 02** | **9-Level Strategy Chain** | **Core: ColumnMapper** | **01** |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---
## Setup

Use Python 3.10+ and run this notebook from `examples/notebooks` in a repository checkout. Select a notebook kernel from the environment containing these packages:

```bash
python -m pip install 'sqlseed[mimesis]==0.2.4' 'sqlseed-cli==0.2.4' jupyterlab
```

For source development, install Core and CLI together as described in the [repository README](../../README.md). This notebook creates its own temporary database and cache. Run cells from top to bottom; the validation helpers raise on partial generation or failed CLI commands.


In [ ]:
# Install the packages listed in Setup into the selected notebook kernel.
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
import os
import tempfile
from pathlib import Path
notebook_temp = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-02-")
work_dir = Path(notebook_temp.name)
os.environ["SQLSEED_CACHE_DIR"] = str(work_dir / "cache")
db_path = build(work_dir / "demo.db")

# Fail visibly if a generation only partially succeeds.
generation_checks = []
def check_result(result, expected_count):
    if result.errors or result.count != expected_count:
        raise RuntimeError(f"{result.table_name}: expected {expected_count}, wrote {result.count}; errors={result.errors}")
    generation_checks.append({"table": result.table_name, "count": result.count, "errors": list(result.errors)})
    print(f"Verified {result.table_name}: {result.count} rows; errors={result.errors}")
    return result

def check_results(results, config_path):
    config = sqlseed.load_config(str(config_path))
    expected = {table.name: table.count for table in config.tables}
    for result in results:
        check_result(result, expected[result.table_name])
    if {result.table_name for result in results} != set(expected):
        raise RuntimeError("Not every configured table produced a result")
    return results

def check_cli(result):
    if result.exit_code != 0:
        raise RuntimeError(result.output) from result.exception
    return result


# Populate base dependencies
with connect(str(db_path)) as orch:
    check_result(orch.fill_table("organizations", count=5, seed=42), 5)
    check_result(orch.fill_table("members", count=20, seed=42), 20)
    check_result(orch.fill_table("projects", count=10, seed=42), 10)
    check_result(orch.fill_table("tags", count=8, seed=42), 8)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Schema Inference | `src/sqlseed/core/schema.py` | `SchemaInferrer` |
| Column Mapping | `src/sqlseed/core/mapper.py` | `ColumnMapper.map_column()` |

> Corresponding architecture diagram: [§3 ColumnMapper 9-Level Strategy Chain](../../docs/architecture.zh-CN.md#3-columnmapper-9-级策略链)

## 1. See the Effect First — The Magic of Zero Config

sqlseed's most powerful feature: **without any configuration**, it can infer the correct data type from column names. Column named `email`? Generate an email. Column named `name`? Generate a name. Column named `created_at`? Generate a timestamp.

See the effect first, then explain the principle:

In [ ]:
# Zero config! sqlseed auto-selects generators based on column names
rows = preview(str(db_path), table="members", count=3)
print(f"{'name':<18s}  {'email':<28s}  {'phone':<18s}  {'org_code':<10s}")
print('-' * 78)
for row in rows:
    print(f"{row.get('name', 'N/A'):<18s}  {row.get('email', 'N/A'):<28s}  {row.get('phone', 'N/A'):<18s}  {row.get('org_code', 'N/A'):<10s}")  # noqa: E501

No mapping config written — sqlseed's `ColumnMapper` auto-completed:

- `name` column → matches `name` rule → generates real name
- `email` column → matches `email` rule → generates email address
- `phone` column → matches `phone` rule → generates phone number
- `member_no` column → matches UNIQUE constraint → generates unique ID

The secret behind this is the **9-level strategy chain** — sqlseed tries each level by priority until it finds a matching generator.

## 2. Strategy Chain Overview

sqlseed's `ColumnMapper` tries to match column names in the following priority order:

| Level | Strategy | Description |
|:----:|------|------|
| 1 | Autoincrement PK | Auto-increment PK auto-skipped |
| 2 | User Config | User explicit config overrides all |
| 3 | Custom Exact Match | Plugin-registered exact rules |
| 4 | Built-in Exact Match | 75 built-in exact match rules |
| 5 | DEFAULT Value | Columns with default skipped or enriched |
| 6 | Custom Pattern Match | Plugin-registered regex rules |
| 7 | Built-in Pattern Match | 29 built-in regex pattern matches |
| 8 | Nullable | Nullable columns skipped or enriched |
| 9 | Type Fallback | 32 SQL types faithful fallback |

Once a level matches, subsequent levels are not executed. Levels 3 and 6 are plugin extension points, see 09-plugin-hooks.ipynb.

## 3. Level 1: Database-Generated Primary Keys

A true SQLite rowid alias or explicitly auto-generated key is skipped by default. `INT PRIMARY KEY`, `WITHOUT ROWID`, and inline `INTEGER PRIMARY KEY DESC` need ordinary generation; the SQL type name alone does not establish autoincrement. An explicit user generator can override an implicit rowid alias.

In [ ]:
from sqlseed import preview

rows = preview(str(db_path), table="members", count=2)
for row in rows:
    print(f"name={row['name']}, email={row['email']}")
print("\nmember_id is an auto-increment PK, preview does not include this column (auto-assigned by SQLite)")

## 4. Level 2: User Config

Use `columns` or YAML to override inferred rules. Explicit database allocation, such as `AUTOINCREMENT`, is handled first; implicit SQLite rowid aliases still accept explicit generators.

In [ ]:
rows = preview(
    str(db_path),
    table="members",
    count=2,
    columns={
        "name": {"generator": "pattern", "params": {"regex": "User-\\d{4}"}},
        "balance": {"generator": "float", "params": {"min_value": 1000.0, "max_value": 5000.0}},
    },
)
for row in rows:
    print(f"name={row['name']}, balance={row['balance']}")
print("\nname and balance are overridden by user config, no longer using auto-inference")

## 5. Level 4: Built-in Exact Match (75 rules)

Exact match is the most commonly used strategy. sqlseed has 75 built-in column-name-to-generator mapping rules.

### Semantic (infer business meaning)

| Column Name | Generator | Params |
|------|--------|------|
| `email` | email | - |
| `phone` | phone | - |
| `name` | name | - |
| `address` | address | - |
| `city` | city | - |
| `country` | country | - |
| `url` / `website` | url | - |
| `password` | password | - |
| `uuid` | uuid | - |

### Numeric name-rule defaults (inspect final schema mapping)

| Column Name | Generator | Params |
|------|--------|------|
| `age` | integer | 18-65 |
| `balance` | float | 0-999999.99 |
| `salary` | float | 3000-100000 |
| `rating` | float | 1.0-5.0 |
| `latitude` | float | -90~90 |

### Enum (fixed options)

| Column Name | Generator | Params |
|------|--------|------|
| `gender` | choice | ["male", "female", "other"] |
| `priority` | choice | ["low", "medium", "high"] |
| `role` | choice | ["admin", "user", "guest"] |
These are mapper defaults, not final guarantees. Schema inference and CHECK adaptation can change them; configure an explicit range when the business value matters. `status` is not a built-in exact-match rule.


In [ ]:
rows = preview(str(db_path), table="members", count=3)
for row in rows:
    print(f"name={row['name']}, email={row['email']}, phone={row['phone']}, balance={row['balance']}")
print("\nname/email/phone/balance all auto-inferred via exact match")

## 6. Level 5: DEFAULT Value

If no earlier explicit/exact rule applies, a DEFAULT column is skipped or enriched. Exact matches take priority over the DEFAULT check. The cell below prints which columns were actually skipped.

In [ ]:
import sqlite3

conn = sqlite3.connect(str(db_path))
cols = conn.execute("PRAGMA table_info(projects)").fetchall()
default_cols = [c[1] for c in cols if c[4] is not None]
print(f"Columns with DEFAULT: {default_cols}")
conn.close()

rows = preview(str(db_path), table="projects", count=2)
print(f"preview output columns: {list(rows[0].keys())}")
skipped = [c for c in default_cols if c not in rows[0]]
print(f"Skipped DEFAULT columns: {skipped}")
print("\nOnly DEFAULT columns not matched by earlier rules are skipped; inspect the printed column lists.")

## 7. Level 7: Built-in Pattern Match (29 regex)

> **Note**: Level 3 (Custom Exact Match) and Level 6 (Custom Pattern Match) are plugin extension points, register custom rules via the `sqlseed_register_column_mappers` Hook. See [09-plugin-hooks.ipynb](09-plugin-hooks.ipynb) Section 10.

When exact match fails, sqlseed uses regex patterns to match column name suffixes:

| Pattern | Generator | Example Column |
|------|--------|----------|
| `.*_id$` | foreign_key_or_integer | `project_id`, `assignee_id` |
| `.*_no$` / `.*_nbr$` | foreign_key_or_integer | `project_no`, `member_no` |
| `.*_at$` | datetime | `created_at`, `due_at` |
| `.*_date$` | date | `birth_date` |
| `^is_.*` / `^has_.*` | boolean | `is_active`, `is_public` |
| `.*_code$` | string (alphanumeric) | `org_code`, `region_code` |
| `.*_name$` | catch_phrase | `org_name`, `file_name` |
| `.*_count$` / `.*_num$` | integer (0-10000) | `task_count`, `item_num` |
| `.*_amount$` / `.*_price$` | float | `total_amount`, `unit_price` |

In [ ]:
# *_no pattern match → foreign_key_or_integer → no FK constraint, falls back to random string (because it's VARCHAR)
# org_code matches *_code → should be string(alphanumeric), but auto-detected FK constraint, smartly upgraded to foreign_key and extracted existing real values from organizations!  # noqa: E501
# *_at pattern match → datetime
rows = preview(str(db_path), table="projects", count=3)
for row in rows:
    pno = str(row['project_no'])[:20]
    code = str(row['org_code'])[:12]
    print(f"project_no={pno:<20s}  org_code={code:<12s}  created_at={row['created_at']}")
print("\nproject_no → *_no pattern → foreign_key_or_integer (no FK, falls back to random string)")
print("org_code → auto-detected FK constraint, smartly upgraded to foreign_key (extracted real values from parent table)")
print("created_at → *_at pattern → datetime")

## 8. Level 8: Nullable

Nullable columns reach the skip/enrich fallback only when earlier naming and explicit rules did not match. A nullable `email` column can still use the email generator.

In [ ]:
conn = sqlite3.connect(str(db_path))
cols = conn.execute("PRAGMA table_info(members)").fetchall()
nullable_cols = [c[1] for c in cols if c[3] == 0 and c[4] is None]
print(f"Nullable columns (no default): {nullable_cols}")
conn.close()

## 9. Level 9: Type Fallback (32 SQL types)

When all naming strategies fail, sqlseed selects a generator based on the column's SQL type:

| SQL Type | Generator | Params |
|----------|--------|------|
| INTEGER | integer | 0-999999 |
| REAL / FLOAT | float | 0-999999 |
| TEXT | string | 5-50 chars |
| VARCHAR(n) | string | 1-n chars |
| BLOB | bytes | 32 bytes |
| BOOLEAN | boolean | - |
| DATE | date | - |
| DATETIME | datetime | - |

In [ ]:
conn = sqlite3.connect(str(db_path))
conn.execute("""CREATE TABLE IF NOT EXISTS demo_fallback (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    info TEXT NOT NULL,
    score_value REAL NOT NULL
)""")
conn.commit()
conn.close()

rows = preview(str(db_path), table="demo_fallback", count=2)
for row in rows:
    info = str(row['info'])[:50]
    print(f"info=\"{info}...\", score_value={row['score_value']:.2f}")
print("\ninfo (TEXT, no column name match) → type fallback → random string (5-50 chars)")
print("score_value (REAL, no column name match) → type fallback → random float")

# Cleanup
conn = sqlite3.connect(str(db_path))
conn.execute("DROP TABLE IF EXISTS demo_fallback")
conn.commit()
conn.close()

## 10. inspect --show-mapping in Practice

Use the CLI's `inspect --show-mapping` command to view the mapping result for each column:

In [ ]:
from click.testing import CliRunner

from sqlseed_cli import cli

runner = CliRunner()
result = check_cli(runner.invoke(cli, ["inspect", str(db_path), "--show-mapping"]))
if result.output.strip():
    print(result.output)

## 11. Custom Mapping Rules (Level 3 / Level 6)

Custom rules can be registered via the `sqlseed_register_column_mappers` plugin Hook. These rules take priority over built-in rules in Level 3 (exact match) and Level 6 (pattern match):

In [ ]:
import pluggy

from sqlseed.plugins.hookspecs import SqlseedHookSpec, hookimpl

pm = pluggy.PluginManager("sqlseed")
pm.add_hookspecs(SqlseedHookSpec)

class CustomMapperPlugin:
    @hookimpl
    def sqlseed_register_column_mappers(self, mapper):
        mapper.register_exact_rule("color", "choice", {"choices": ["red", "green", "blue"]})
        mapper.register_pattern_rule(r".*_color$", "choice", {"choices": ["#ff0000", "#00ff00", "#0000ff"]})

pm.register(CustomMapperPlugin())
print("Custom mapper plugin registered")
print("  Exact rule: 'color' → choice([red, green, blue])")
print("  Pattern rule: '*_color' → choice([#ff0000, #00ff00, #0000ff])")
from sqlseed.core.mapper import ColumnMapper
from sqlseed.database import ColumnInfo
mapper = ColumnMapper()
pm.hook.sqlseed_register_column_mappers(mapper=mapper)
spec = mapper.map_column(ColumnInfo(name="color", type="TEXT", nullable=False, default=None, is_primary_key=False, is_autoincrement=False))
assert spec.generator_name == "choice"
assert spec.params["choices"] == ["red", "green", "blue"]
print("Applied custom mapping:", spec)


## 🎯 enrich Mode: __enrich__ Behavior for Level 5 and Level 8

Normally, columns in Level 5 (with DEFAULT values) and Level 8 (nullable) are **skipped**. But when `enrich=True`, these columns enter `__enrich__` mode:

- **DEFAULT columns**: if identified as enum columns by EnrichmentEngine, generate meaningful enum values
- **Nullable columns**: if identified as enum columns, generate enum values; otherwise generate based on null_ratio

### EnrichmentEngine Enum Column Detection

EnrichmentEngine uses 19 column name patterns to detect enum columns:

| Pattern | Example Column |
|---|---|
| `*_status` | order_status, project_status |
| `*_type` | user_type, file_type |
| `is_*` | is_active, is_public |
| `has_*` | has_permission |
| `*_level` | priority_level, access_level |
| `*_category` | product_category |
| `*_flag` | feature_flag |
| `*_mode` | payment_mode |
| ... | 19 patterns in total |

Additionally, it uses **cardinality ratio** (distinct_count / total_rows < 0.3) and **small integer types** (INT8/INT16/TINYINT/SMALLINT) to assist in judgment.

In [ ]:
from sqlseed.core.enrichment import EnrichmentEngine

print("EnrichmentEngine enum column name patterns (19):")
for i, pattern in enumerate(EnrichmentEngine.ENUM_NAME_PATTERNS, 1):
    print(f"  {i:2d}. {pattern}")

print(f"\nSmall integer types: {EnrichmentEngine.SMALL_INT_TYPES}")

## 🔗 foreign_key_or_integer Resolution

`*_id` and `*_no` patterns are resolved using real FK metadata and registered parent-value pools. Ordinary matching names alone do not guarantee a relation. Use explicit `ColumnAssociation` rules for relationships without declared FKs, as demonstrated in notebook 04/06. Without an applicable relation, generation falls back to the column type.


In [ ]:
with sqlseed.connect(str(db_path)) as orch:
    col_info = orch.get_column_info("members")
    fk_info = orch.get_foreign_keys("members")

    fk_columns = {fk.column for fk in fk_info}
    print("members table column mapping analysis:")
    for col in col_info:
        if col.name.endswith(("_id", "_no")):
            is_fk = col.name in fk_columns
            gen_type = "foreign_key" if is_fk else "integer/string"
            print(f"  {col.name}: {gen_type} (FK={is_fk})")

## 12. Summary

| Level | Strategy | Rule Count | Priority |
|:----:|------|:------:|:------:|
| 1 | Autoincrement PK | - | Highest |
| 2 | User Config | Unlimited | Very High |
| 3 | Custom Exact Match | Plugin-registered | High |
| 4 | Built-in Exact Match | 75 | High |
| 5 | DEFAULT Value | - | Medium-High |
| 6 | Custom Pattern Match | Plugin-registered | Medium |
| 7 | Built-in Pattern Match | 29 | Medium |
| 8 | Nullable | - | Medium-Low |
| 9 | Type Fallback | 32 | Lowest |

**Key Insights**:
- Naming is more important than type — `email VARCHAR(128)` generates an email, not a random string
- User config can override all auto-inference
- Plugins can inject custom rules via Level 3/6, taking priority over built-in rules
- `inspect --show-mapping` is the primary tool for debugging mapping issues

**Next**: [03-generators.ipynb](03-generators.ipynb) — Learn about 36 generators and the Provider system

In [ ]:
# Verify exact database totals after the full notebook, plus every declared FK.
import sqlite3
expected_counts = {'organizations': 5, 'members': 20, 'projects': 10, 'tasks': 0, 'tags': 8, 'reviews': 0}
with sqlite3.connect(str(db_path)) as verification_db:
    actual_counts = {
        table: verification_db.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
        for table in expected_counts  # Fixed tutorial table names.
    }
    assert actual_counts == expected_counts, (actual_counts, expected_counts)
    fk_errors = verification_db.execute("PRAGMA foreign_key_check").fetchall()
    assert not fk_errors, fk_errors
print("Verified database row counts:", actual_counts)
print("Database FK check:", fk_errors)
print("Verified fill operations:", len(generation_checks))
